**imports and path setup**

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import numpy as np


In [ ]:
# paths
classification_data = "../data/raw/classification"
segmentation_data = "../data/raw/segmentation/kaggle_3m"

# checks is the path exists or not 

print(Path(classification_data).exists())
print(Path(segmentation_data).exists())

**Explore classification dataset structure**

In [ ]:
# move to training folder
train_path = Path(classification_data) / "Training"

# This lists the subfolders inside Training/ — 
# should be your 4 class names (glioma, meningioma, pituitary, notumor or similar exact spelling).

class_folders = list(train_path.iterdir())
print(class_folders)

In [ ]:
# checking for imbalance data

for folder in class_folders:
    name = folder.name
    count = len(list(folder.iterdir()))
    print(f"Folder name : {name} with image count : {count}")

# class disteribution checking

In [ ]:
records = []

for split in ["Training","Testing"]:
    split_path = Path(classification_data) / split
    for folder in split_path.iterdir():
        count = len(list(folder.iterdir()))
        records.append({"split": split,"class": folder.name,"count":count})
df = pd.DataFrame(records)
df
    

In [ ]:
pivot = df.pivot(index = "class", columns = "split", values = "count")
pivot.plot(kind = "bar")
plt.title("Class Distribution: Training vs Testing")
plt.ylable("Image Count")
plt.show()

# Segmentation dataset — explore one patient folder
looking inside a single patient folder to understand the file naming pattern

In [ ]:
seg_path = Path(segmentation_data)
patient_folders = [p for p in seg_path.iterdir() if p.is_dir()]
print(len(patient_folders))

In [ ]:
first_patient = patient_folders[0]
print(first_patient.name)
print(list(first_patient.iterdir()))

# Loading and displaying one image + its mask side-by-side

In [ ]:
# img_path = first_patient / "TCGA_CS_4941_19960909_1.tif"
# mask_path = first_patient / "TCGA_CS_4941_19960909_1_mask.tif"

img_path = first_patient / "TCGA_CS_4941_19960909_11.tif"
mask_path = first_patient / "TCGA_CS_4941_19960909_11_mask.tif"

img = Image.open(img_path)
mask = Image.open(mask_path)

fig , axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img)
axes[0].set_title("MRI Image")
axes[1].imshow(mask, cmap = "gray")
axes[1].set_title("Mask")

plt.show()

In [ ]:
# checking if the 1st patient even have tumor or not

mask_files = sorted(first_patient.glob("*_mask.tif"))

for mask_file in mask_files:
    mask_arr = np.array(Image.open(mask_file))
    if mask_arr.max() > 0:
        print(mask_file.name," Has Tumor")

# Image-mask pairing verification (across ALL 110 patients)

In [ ]:
mismatches = []

for patient in patient_folders:
    all_tifs = list(patient.glob("*.tif"))
    masks = [f for f in all_tifs if "_mask" in f.name]
    images = [f for f in all_tifs if "_mask" not in f.name]
    if len(masks) != len(images):
        mismatches.append((patient.name, len(images), len(masks)))
print(f"Total patients checked: {len(patient_folders)}")
print(f"Mismatches found: {len(mismatches)}")
print(mismatches)

# Corrupt file check

In [ ]:
def check_corrupt_files(root_folder, pattern="*.jpg"):
    root = Path(root_folder)
    corrupt_files = []
    all_files = list(root.rglob(pattern))
    
    for file in all_files:
        try:
            img = Image.open(file)
            img.verify()
        except Exception as e:
            corrupt_files.append((file, str(e)))
    
    print(f"Checked: {len(all_files)} files")
    print(f"Corrupt: {len(corrupt_files)}")
    return corrupt_files

print("corrupt file in classification")
corrupt_classification = check_corrupt_files(classification_data, "*.jpg")

print("corrupt file in segmentation")

corrupt_segmentation = check_corrupt_files(segmentation_data, "*.tif")


# Dimension & format consistency check

In [ ]:
def check_dimensions(root_folder, pattern="*.jpg"):
    root = Path(root_folder)
    sizes = set()
    modes = set()
    
    for file in root.rglob(pattern):
        img = Image.open(file)
        sizes.add(img.size)
        modes.add(img.mode)
    
    print(f"Unique sizes: {sizes}")
    print(f"Unique modes: {modes}")

print("Images in jpg")
check_dimensions(classification_data, "*.jpg")
print("images in tif")
check_dimensions(segmentation_data, "*.tif")

In [ ]:
img_modes = set()
mask_modes = set()

for file in Path(segmentation_data).rglob("*.tif"):
    mode = Image.open(file).mode
    if "_mask" in file.name:
        mask_modes.add(mode)
    else:
        img_modes.add(mode)

print("Image modes:", img_modes)
print("Mask modes:", mask_modes)

# 01 — Data Exploration Summary

## Classification Dataset (Kaggle Brain Tumor MRI — Nickparvar)
- 4 classes: `glioma`, `meningioma`, `notumor`, `pituitary`
- Perfectly balanced: 1400 images/class in Training, 400 images/class in Testing (7200 total)
- **No class imbalance handling needed** for training
- Inconsistent image dimensions (hundreds of unique sizes) and mixed color modes (`L`, `P`, `RGB`, `RGBA`)
- **Preprocessing requirement:** resize to fixed input size (224×224) + force-convert all images to RGB before training
- 0 corrupt files found (7200 checked)

## Segmentation Dataset (LGG MRI Segmentation — Buda)
- 110 patients, uniform image size 256×256
- Images are always RGB (multi-sequence MRI channels, not true color), masks are always grayscale (`L`)
- Naming convention: mask = `{base_filename}_mask.tif`
- Image-mask pairing verified: **0 mismatches across all 110 patients**
- 0 corrupt files found (7858 checked)
- Most slices per patient have empty (tumor-free) masks — only a subset of slices show visible tumor; this is expected, not a data issue. Model must be trained on empty masks too so it learns to correctly predict "no tumor."
- License: CC-BY-NC-SA-4.0 — non-commercial use only; flag if deployment plans change to commercial

## Key Takeaways for Weekend 2+
- Classification pipeline needs resize + RGB normalization step
- Segmentation pipeline needs no resizing, but must explicitly include no-tumor slices in training set
- No weighted loss required for classifier (balanced classes)
- Dice/BCE combo loss planned for segmenter per class imbalance at pixel level (tumor vs background)